# Day 18 — Build a Mini Agent from Scratch

**Week 3: LLM Inference → Agent → Graph × LLM**

Today we move from *understanding* ReAct to *implementing* the core agent loop ourselves.

> **Goal:** build a minimal agent without LangChain or any agent framework.

By the end of today, you should be able to explain and implement:

$$
\text{Task}
\rightarrow
\text{LLM decision}
\rightarrow
\text{Action}
\rightarrow
\text{Tool}
\rightarrow
\text{Observation}
\rightarrow
\text{State update}
\rightarrow
\text{LLM decision}
\rightarrow \cdots
\rightarrow
\text{Final}
$$

The point is **not** to build a production agent. The point is to understand the computational structure clearly enough that later agent frameworks no longer look mysterious.

## 0. What changed from Day 17?

Yesterday's toy code had a hidden shortcut:

```python
steps = [
    ("Thought", ...),
    ("Action", ...),
    ("Thought", ...),
    ("Final", ...)
]
```

The entire trajectory was written by us in advance. That is **not yet a real agent**.

Today we remove that hard-coded `steps` list. The next step will be produced dynamically from the current task and trajectory.

```text
Day 17:
Human writes the whole trajectory → Python executes it

Day 18:
Agent reads current state → chooses next step dynamically
```

## 1. The minimal agent architecture

A useful mental model is:

$$
\boxed{\text{Agent} \approx \text{LLM} + \text{Tools} + \text{State} + \text{Control Loop}}
$$

This is a simplified systems view, not a strict mathematical definition.

- **LLM / policy**: decides what to do next.
- **Tool**: performs an external operation.
- **Observation**: the tool/environment result.
- **State / trajectory**: stores information needed for future decisions.
- **Control loop**: decides whether to continue acting or terminate with a final answer.

The most important idea today:

> The agent is not a single model call. It is a program that repeatedly invokes a decision model and the environment.

## Mandatory Question 1

Why is the following code **not** a real dynamic agent?

```python
steps = [
    ("Thought", "I should search X"),
    ("Action", ("search", "X")),
    ("Thought", "Now I know the answer"),
    ("Final", "...")
]
```

Answer in your own English.

answer : Because the steps should be generated automatically by LLM instead of manual, and in reality the steps is produced based on trajactory
Because the steps are manually hard-coded in advance. In a real dynamic agent, the next action should be generated by the LLM based on the current task and trajectory, including previous actions and observations.

## 2. First build the environment: tools

We will use tiny local tools so the notebook is reproducible and does not require any API key.

A tool is just a callable interface exposed to the agent.

We will build three tools:

1. `lookup` — retrieve a fact from a small knowledge base.
2. `calculator` — perform a restricted arithmetic calculation.
3. `graph_lookup` — traverse one edge in a tiny knowledge graph.

The agent does **not** directly access the underlying data structures. It must go through tools.

In [2]:
from typing import Any, Dict, List
import ast
import operator as op
import json

In [3]:
KNOWLEDGE = {
    "react_venue": "ICLR 2023",
    "iclr_2023_city": "Kigali",
    "kigali_country": "Rwanda",
    "france_capital": "Paris",
    "paris_country": "France",
}

GRAPH = {
    ("ReAct", "published_at"): "ICLR 2023",
    ("ICLR 2023", "held_in"): "Kigali",
    ("Kigali", "located_in"): "Rwanda",
}

### A safe mini calculator

Avoid using unrestricted `eval()` for arbitrary input. We support only a small set of arithmetic operators so that the example remains transparent.

In [5]:
_ALLOWED_OPS = {
    ast.Add: op.add,
    ast.Sub: op.sub,
    ast.Mult: op.mul,
    ast.Div: op.truediv,
    ast.Pow: op.pow,
    ast.USub: op.neg,
}

def _eval_ast(node):
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_OPS:
        return _ALLOWED_OPS[type(node.op)](_eval_ast(node.left), _eval_ast(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_OPS:
        return _ALLOWED_OPS[type(node.op)](_eval_ast(node.operand))
    raise ValueError("Unsupported expression")

def calculator(expr: str):
    tree = ast.parse(expr, mode="eval")
    return _eval_ast(tree.body)

In [37]:
def lookup(key: str):
    return KNOWLEDGE.get(key, "NOT_FOUND")

def graph_lookup(entity: str, relation: str):
    return GRAPH.get((entity, relation), "NOT_FOUND")

def tool_call(tool_name: str, args: Dict[str, Any]):
    if tool_name == "lookup":
        return lookup(args["key"])
    if tool_name == "graph_lookup":
        return graph_lookup(args["entity"], args["relation"])
    if tool_name == "calculator":
        return calculator(args["expression"])
    return f"UNKNOWN_TOOL: {tool_name}"

In [10]:
print(tool_call("lookup", {"key": "react_venue"}))
print(tool_call("calculator", {"expression": "23 * 17"}))
print(tool_call("graph_lookup", {"entity": "Kigali", "relation": "located_in"}))
print(tool_call("Wednesday", {"key": "react_venue"}))

ICLR 2023
UNKNOWN_TOOL: calculator
Rwanda
UNKNOWN_TOOL: Wednesday


## Mandatory Question 2

In this notebook:

- What is the **environment**?
- What is the **tool**?
- What is an **Action**?
- Where does the **Observation** come from?

Do not just copy definitions. Explain the data flow.

- environment: KNOWLEDGE,GRAPH
- tool: lookup,graph_lookup,calculator
- Action: a tool call selected by the agent, including the tool name and its arguments
- Observation: the result returned by the environment/tool after executing an Action

## 3. Define the agent state

A real agent needs persistent state.

We will store:

- the original task,
- the trajectory,
- the current step count,
- whether the agent has terminated,
- the final answer.

A trajectory entry may look like:

```python
{"type": "action", "tool": "lookup", "args": {...}}
{"type": "observation", "content": "ICLR 2023"}
{"type": "final", "content": "Rwanda"}
```

For implementation clarity, we will **not require explicit hidden Thought text**. The decision model can simply emit either an `action` or a `final` decision.

This is closer to how many practical agent systems are structured: internal reasoning need not be exposed as a user-visible trace.

In [12]:
def make_state(task: str) -> Dict[str, Any]:
    return {
        "task": task,
        "trajectory": [],
        "step": 0,
        "done": False,
        "final_answer": None,
    }

## 4. Define the decision interface

The agent's decision model should return one of two structures.

### Action

```python
{
    "type": "action",
    "tool": "lookup",
    "args": {"key": "react_venue"}
}
```

### Final

```python
{
    "type": "final",
    "answer": "Rwanda"
}
```

This gives us an important abstraction:

$$
\text{Agent Decision} \in \{\text{Action}, \text{Final}\}
$$

The control loop does not need to know *how* the model made the decision. It only needs a structured interface.

## 5. A deterministic mock LLM

Before connecting a real language model, we first use a deterministic policy.

Why? Because we want to debug the **agent architecture** independently from model uncertainty.

> Isolate one source of complexity at a time.

In [13]:
def mock_llm(task: str, trajectory: List[Dict[str, Any]]) -> Dict[str, Any]:
    observations = [x["content"] for x in trajectory if x["type"] == "observation"]

    if "ICLR 2023" in task and "country" not in task.lower():
        if "Kigali" not in observations:
            return {"type": "action", "tool": "lookup", "args": {"key": "iclr_2023_city"}}
        return {"type": "final", "answer": "Kigali"}

    if "ReAct" in task and "country" in task.lower():
        if "ICLR 2023" not in observations:
            return {"type": "action", "tool": "graph_lookup", "args": {"entity": "ReAct", "relation": "published_at"}}
        if "Kigali" not in observations:
            return {"type": "action", "tool": "graph_lookup", "args": {"entity": "ICLR 2023", "relation": "held_in"}}
        if "Rwanda" not in observations:
            return {"type": "action", "tool": "graph_lookup", "args": {"entity": "Kigali", "relation": "located_in"}}
        return {"type": "final", "answer": "Rwanda"}

    if "23 * 17" in task:
        if 391 not in observations:
            return {"type": "action", "tool": "calculator", "args": {"expression": "23 * 17"}}
        return {"type": "final", "answer": "391"}

    return {"type": "final", "answer": "I do not know how to solve this task."}

## 6. The actual agent loop

This is the most important code cell today.

Read it until you can map every line to the conceptual loop:

```text
state
  ↓
decision model
  ↓
Action ──→ Tool ──→ Observation ──→ state
  ↑                                  │
  └──────────────────────────────────┘

or

decision model
  ↓
Final
  ↓
stop
```

In [25]:
def run_agent(task: str, decision_model, max_steps: int = 8, verbose: bool = True):
    state = make_state(task)

    while not state["done"] and state["step"] < max_steps:
        state["step"] += 1

        decision = decision_model(state["task"], state["trajectory"])

        if decision["type"] == "action":
            action_record = {
                "type": "action",
                "tool": decision["tool"],
                "args": decision["args"],
            }
            state["trajectory"].append(action_record)

            observation = tool_call(decision["tool"], decision["args"])

            observation_record = {
                "type": "observation",
                "content": observation
            }
            state["trajectory"].append(observation_record)

            if verbose:
                print(f"[step {state['step']}] ACTION:", action_record)
                print(f"[step {state['step']}] OBSERVATION:", observation)

        elif decision["type"] == "final":
            state["done"] = True
            state["final_answer"] = decision["answer"]
            state["trajectory"].append({"type": "final", "content": decision["answer"]})

            if verbose:
                print(f"[step {state['step']}] FINAL:", decision["answer"])
        else:
            raise ValueError(f"Unknown decision type: {decision['type']}")

    if not state["done"]:
        state["final_answer"] = "STOPPED: max_steps reached"

    return state    

In [26]:
state = run_agent(
    task="In which country was the conference that published ReAct held?",
    decision_model=mock_llm,
)

[step 1] ACTION: {'type': 'action', 'tool': 'graph_lookup', 'args': {'entity': 'ReAct', 'relation': 'published_at'}}
[step 1] OBSERVATION: ICLR 2023
[step 2] ACTION: {'type': 'action', 'tool': 'graph_lookup', 'args': {'entity': 'ICLR 2023', 'relation': 'held_in'}}
[step 2] OBSERVATION: Kigali
[step 3] ACTION: {'type': 'action', 'tool': 'graph_lookup', 'args': {'entity': 'Kigali', 'relation': 'located_in'}}
[step 3] OBSERVATION: Rwanda
[step 4] FINAL: Rwanda


In [27]:
state["trajectory"]

[{'type': 'action',
  'tool': 'graph_lookup',
  'args': {'entity': 'ReAct', 'relation': 'published_at'}},
 {'type': 'observation', 'content': 'ICLR 2023'},
 {'type': 'action',
  'tool': 'graph_lookup',
  'args': {'entity': 'ICLR 2023', 'relation': 'held_in'}},
 {'type': 'observation', 'content': 'Kigali'},
 {'type': 'action',
  'tool': 'graph_lookup',
  'args': {'entity': 'Kigali', 'relation': 'located_in'}},
 {'type': 'observation', 'content': 'Rwanda'},
 {'type': 'final', 'content': 'Rwanda'}]

## Mandatory Question 3

Explain the control flow of `run_agent()` in your own words.

Your answer must mention:

- where the next decision comes from,
- when a tool is executed,
- how the Observation is inserted into state,
- when the loop terminates.

回答：
the next decision comes from the decision_model, which takes the current task and trajectory as input. In this notebook, the decision model is mock_llm. If the model returns an Action, the agent executes the corresponding tool with tool_call() and gets an Observation. The Observation is then appended to the trajectory so that it can affect the next decision. The loop continues until the model returns a Final answer or the agent reaches max_steps.

## 7. Why `max_steps` matters

An agent needs a termination mechanism.

Possible termination conditions include:

- the model emits `Final`,
- the environment reaches a terminal state,
- a maximum number of steps is reached,
- a token / latency / cost budget is exhausted,
- an external verifier accepts the answer.

Without a stopping rule, an agent can enter:

```text
Action → Observation → Action → Observation → ...
```

forever.

This is not merely a coding issue. It is a real **agent reliability** problem.

In [28]:
def looping_model(task, trajectory):
    return {"type": "action", "tool": "lookup", "args": {"key": "react_venue"}}

loop_state = run_agent(
    "keep searching forever",
    decision_model=looping_model,
    max_steps=3,
)
print(loop_state["final_answer"])

[step 1] ACTION: {'type': 'action', 'tool': 'lookup', 'args': {'key': 'react_venue'}}
[step 1] OBSERVATION: ICLR 2023
[step 2] ACTION: {'type': 'action', 'tool': 'lookup', 'args': {'key': 'react_venue'}}
[step 2] OBSERVATION: ICLR 2023
[step 3] ACTION: {'type': 'action', 'tool': 'lookup', 'args': {'key': 'react_venue'}}
[step 3] OBSERVATION: ICLR 2023
STOPPED: max_steps reached


## Mandatory Question 4

Why is a stopping condition part of the **agent architecture**, rather than just a minor implementation detail?

Answer: Because it determines when the agent should terminate its loop. Without a stopping condition, the agent may keep reasoning and using tools indefinitely, increasing cost and latency.

## 8. Failure mode: wrong tool selection

A model can reason correctly at a high level but still choose the wrong tool.

```text
Task: calculate 23 × 17

Correct high-level intention:
"I need an exact calculation."

Wrong Action:
lookup("23 * 17")

Observation:
NOT_FOUND
```

The problem is no longer purely "reasoning quality". The interface between reasoning and action can fail.

In [40]:
def wrong_tool_model(task, trajectory):
    observations = [x["content"] for x in trajectory if x["type"] == "observation"]
    if not observations:
        return {"type": "action", "tool": "lookup", "args": {"key": "23 * 17"}}
    return {"type": "final", "answer": f"I got: {observations[-1]}"}

wrong_state = run_agent(
    "Calculate 23 * 17",
    decision_model=wrong_tool_model,
)

[step 1] ACTION: {'type': 'action', 'tool': 'lookup', 'args': {'key': '23 * 17'}}
[step 1] OBSERVATION: NOT_FOUND
[step 2] FINAL: I got: NOT_FOUND


## 9. Failure mode: noisy / misleading observations

ReAct does **not** eliminate error propagation.

External observations can help correct internal mistakes, but they also create new failure paths:

```text
correct reasoning
    ↓
wrong tool / noisy observation
    ↓
incorrect state update
    ↓
incorrect next decision
```

The agent can therefore fail because of model reasoning, tool selection, argument generation, tool execution, noisy external data, observation interpretation, state management, or stopping logic.

## Mandatory Question 5

ReAct introduces external grounding. Why does that **not** imply that ReAct eliminates hallucination or error propagation?

Answer: External grounding provides additional information, but it does not guarantee that the information is correct or correctly interpreted. A wrong tool, noisy observation, or incorrect state update can still affect later decisions and cause error propagation.

## 10. A better architecture: tool registry

Hard-coding many `if tool_name == ...` branches does not scale. A cleaner design uses a **tool registry**.

In [41]:
TOOL_REGISTRY = {
    "lookup": lookup,
    "calculator": calculator,
    "graph_lookup": graph_lookup,
}

def registered_tool_call(tool_name: str, args: Dict[str, Any]):
    if tool_name not in TOOL_REGISTRY:
        return f"UNKNOWN_TOOL: {tool_name}"

    tool = TOOL_REGISTRY[tool_name]

    if tool_name == "lookup":
        return tool(args["key"])
    if tool_name == "calculator":
        return tool(args["expression"])
    if tool_name == "graph_lookup":
        return tool(args["entity"], args["relation"])

The important idea is not the Python dictionary itself.

> The agent receives an explicit **action space**.

The model does not have arbitrary access to your machine. It can only select from tools that the runtime exposes.

This separation becomes critical for safety, permissions, reliability, reproducibility, debugging, and tool routing.

## 11. Tool schema: make actions structured

A real agent should not rely on vague free-form text like:

```text
"Maybe search something about ReAct."
```

A structured action is much easier to validate:

```json
{
  "tool": "graph_lookup",
  "args": {
    "entity": "ReAct",
    "relation": "published_at"
  }
}
```

This creates a clean interface between:

$$
\text{LLM decision} \quad\text{and}\quad \text{program execution}
$$

Think of tool calling as a small **API contract** between the model and the runtime.

## Mandatory Question 6

Why is a structured tool schema better than asking the model to output arbitrary natural-language commands?

Answer: A structured tool schema provides a clear and machine-readable interface between the LLM and the runtime. It reduces ambiguity, allows validation of tool names and arguments, and makes tool execution more reliable and easier to extend.

## 12. From mock LLM to real LLM

Our architecture is already agent-like. The only fake component is:

```python
mock_llm(...)
```

A real implementation would replace it with something conceptually like:

```python
def real_llm(task, trajectory, tools):
    prompt = build_context(task, trajectory, tools)
    raw_output = call_model(prompt)
    decision = parse_and_validate(raw_output)
    return decision
```

Then the **same control loop** can remain unchanged.

```text
decision model
      ↓
structured decision
      ↓
agent runtime
      ↓
tool execution
```

The runtime should not care whether the decision came from a deterministic rule, an LLM, a fine-tuned policy, a planner, a graph model, or a hybrid system.

## 13. Context construction

The next model call must somehow receive the useful parts of the previous trajectory.

In [ ]:
def serialize_trajectory(trajectory):
    lines = []
    for item in trajectory:
        if item["type"] == "action":
            lines.append(f"ACTION {item['tool']} {json.dumps(item['args'])}")
        elif item["type"] == "observation":
            lines.append(f"OBSERVATION {item['content']}")
        elif item["type"] == "final":
            lines.append(f"FINAL {item['content']}")
    return "\n".join(lines)

print(serialize_trajectory(state["trajectory"]))

In many simple agents, the next LLM call receives something conceptually similar to:

```text
TASK:
...

TRAJECTORY:
ACTION ...
OBSERVATION ...
ACTION ...
OBSERVATION ...

AVAILABLE TOOLS:
...
```

The model then predicts the next structured decision. This is why context length matters for long agent trajectories.

## Mandatory Question 7

What is the difference between:

- **trajectory as a log**, and
- **trajectory as model context**?

Why can a long trajectory become a systems problem?

Answer: Trajectory as a log is the interaction history stored by the agent runtime. Trajectory as model context is the selected and serialized part of that history that is passed to the LLM for the next decision. A long trajectory can become a systems problem because it increases context length, which raises token cost, memory/KV-cache usage, and inference latency. It may also introduce irrelevant or noisy information into the model context.

## 14. Controlled experiment: remove the Observation

Now connect today's agent work to the ablation mindset from Week 2.

> What happens if the Action is executed but the Observation is **not** written back into the trajectory?

This isolates the value of the feedback loop.

In [42]:
def run_agent_without_observation(task: str, decision_model, max_steps: int = 4):
    state = make_state(task)

    while not state["done"] and state["step"] < max_steps:
        state["step"] += 1
        decision = decision_model(state["task"], state["trajectory"])

        if decision["type"] == "action":
            state["trajectory"].append({
                "type": "action",
                "tool": decision["tool"],
                "args": decision["args"],
            })
            _ = tool_call(decision["tool"], decision["args"])
            # Intentionally discard the observation.

        elif decision["type"] == "final":
            state["done"] = True
            state["final_answer"] = decision["answer"]

    if not state["done"]:
        state["final_answer"] = "STOPPED: max_steps reached"

    return state

ablated_state = run_agent_without_observation(
    "In which country was the conference that published ReAct held?",
    decision_model=mock_llm,
)

print(ablated_state["final_answer"])
print(ablated_state["trajectory"])

STOPPED: max_steps reached
[{'type': 'action', 'tool': 'graph_lookup', 'args': {'entity': 'ReAct', 'relation': 'published_at'}}, {'type': 'action', 'tool': 'graph_lookup', 'args': {'entity': 'ReAct', 'relation': 'published_at'}}, {'type': 'action', 'tool': 'graph_lookup', 'args': {'entity': 'ReAct', 'relation': 'published_at'}}, {'type': 'action', 'tool': 'graph_lookup', 'args': {'entity': 'ReAct', 'relation': 'published_at'}}]


### Interpretation

The agent repeatedly lacks the information it needs to advance.

$$
\boxed{\text{Observation updates state} \rightarrow \text{state changes future decisions}}
$$

The tool call alone is not enough. The result must become usable information for the next decision.

## Mandatory Question 8

Why is the previous experiment a meaningful ablation?

What component was removed, and what causal claim does the result support?

Answer: The removed component is the Observation → state update step. Since the agent then repeats the same action and cannot make progress, the result supports the claim that Observation feedback is necessary for the agent to update its state and change future decisions.

## 15. Agent workload vs ordinary LLM workload

An ordinary single LLM request may look like:

```text
Prompt → Prefill → Decode → Answer
```

An agent task may look like:

```text
LLM request 1
    ↓
tool call
    ↓
observation
    ↓
LLM request 2
    ↓
tool call
    ↓
observation
    ↓
LLM request 3
    ↓
...
```

Therefore the total task cost is roughly:

$$
\text{Total Agent Cost}
\approx
\sum_{t=1}^{T}
\left(
\text{LLM Cost}_t
+
\text{Tool Cost}_t
+
\text{State/Context Cost}_t
\right)
$$

where $T$, the number of agent steps, is often **data-dependent and unknown in advance**.

## Mandatory Question 9

Why does agentic execution create a more dynamic serving workload than ordinary single-turn generation?

Connect your answer to at least three of:

- variable number of LLM calls,
- variable tool latency,
- variable observation length,
- growing context,
- unpredictable termination,
- scheduling / batching.

Answer: Agent workloads are more dynamic because the number of LLM calls, tool latency, observation length, and context size can all vary during execution. Since each observation can change the next decision, the total number of steps and termination time are unpredictable, which makes memory management, scheduling, and batching more difficult.

## 16. Graph connection: why this matters for your research direction

The `graph_lookup` tool is not accidental.

A graph-aware agent could use actions such as:

```text
get_neighbors(entity)
follow_relation(entity, relation)
retrieve_subgraph(query)
find_path(source, target)
rank_candidate_nodes(...)
```

Then the loop becomes:

```text
LLM reasoning
    ↓
Graph Action
    ↓
Graph retrieval / traversal
    ↓
Subgraph Observation
    ↓
LLM reasoning
    ↓
next Graph Action or Final
```

This is one bridge from:

$$
\text{Graph Learning}
\rightarrow
\text{Agent}
\rightarrow
\text{Graph RAG / KG Agent / Structured Reasoning}
$$

Later in Week 3, we will study this connection explicitly.

## Mandatory Question 10

Suppose an Agent can call:

```python
graph_lookup(entity, relation)
```

What new failure modes appear compared with ordinary text generation?

Give at least three examples.

Answer: New failure modes include selecting the wrong entity, selecting the wrong relation, querying an edge that does not exist, or failing to resolve the entity name correctly. The Agent may also choose the wrong tool or generate invalid arguments.

# Day 18 Final Recap

Answer these without looking back if possible.

1. What four components form the minimal agent architecture?
2. Why is the Day 17 hard-coded `steps` example not a real dynamic agent?
3. What is the difference between Action and Tool execution?
4. Where does Observation come from, and where should it go?
5. Why must the agent maintain state / trajectory?
6. What are the two main output types of the decision model?
7. Why do we need a termination condition?
8. Why is structured tool calling better than free-form action text?
9. What happens if observations are discarded?
10. Why are agent workloads more dynamic than ordinary LLM serving workloads?
11. In one sentence, define a Mini Agent.

# Related Learning Materials

You do **not** need to read all of these today.

## Required today

### ReAct — Yao et al.
**ReAct: Synergizing Reasoning and Acting in Language Models**

- arXiv: https://arxiv.org/abs/2210.03629
- PDF: https://arxiv.org/pdf/2210.03629

You already studied the core idea on Day 17. Today, use it only as conceptual background for the implementation.

## Recommended next

### MRKL Systems
**MRKL Systems: A modular, neuro-symbolic architecture that combines large language models, external knowledge sources and discrete reasoning**

- arXiv: https://arxiv.org/abs/2205.00445

**Why it matters:** it helps you think about LLMs as routers/controllers over multiple external modules.

Focus on modular tools, routing, and external symbolic modules. Do **not** deep-read it now.

### Toolformer
**Toolformer: Language Models Can Teach Themselves to Use Tools**

- arXiv: https://arxiv.org/abs/2302.04761

**Why it matters:** ReAct asks *how an agent alternates reasoning and acting*. Toolformer asks a different question: *how can a model learn when and how to call tools?*

Focus on tool-use learning, when a tool call is useful, and tool-call supervision.

### Reflexion
**Reflexion: Language Agents with Verbal Reinforcement Learning**

- arXiv: https://arxiv.org/abs/2303.11366

**Why it matters:** it extends the basic agent loop with self-evaluation / reflection across attempts.

Focus on feedback, reflection, and memory across trajectories. Read this only after the minimal loop feels natural.

## Optional systems-side connection

Revisit Day 16 concepts later:

- continuous batching,
- KV-cache management,
- PagedAttention,
- latency vs throughput.

The research question becomes:

> How should an inference system efficiently serve **dynamic agent workloads**, not just isolated chat requests?

## Optional graph-side preview

Do not read a new graph-agent paper today. For now, only keep this taxonomy in mind:

```text
LLM for Graph
Graph for LLM
Graph RAG
Knowledge-Graph Agent
Graph Reasoning
Graph-structured Memory
```

Days 19–20 will build this map carefully.

# Suggested Schedule for Today

### Part A — Understand the architecture
Sections 1–6  
**Target:** 45–60 minutes

### Part B — Run and modify the Mini Agent
Sections 7–14  
**Target:** 60–90 minutes

Try at least two modifications:

- add one new tool,
- add one new task that requires multiple tool calls.

### Part C — Answer mandatory questions
**Target:** 30–45 minutes

Write your answers in English first. Clarity matters more than sophisticated vocabulary.

### Part D — Optional reading
Read only the abstract + architecture idea of **MRKL** or **Toolformer** if you still have energy.

Do not turn Day 18 into another full paper-reading day.

# Day 18 Deliverable

By the end of today, your deliverable should be:

```text
day18_mini_agent.ipynb
```

with:

- all code cells executed,
- at least one multi-step agent trajectory,
- one custom tool or custom task added by you,
- all 10 mandatory questions answered,
- the final recap completed in English.

If you can explain `run_agent()` without looking at the notebook, Day 18 is complete.